# 📘 Module 15.6 — Regularized MLP — Complete Notes

## 1. Why Regularize an MLP? 🔥🔥🔥

A model can perform extremely well on training data but poorly on unseen data.

Example:

```text
Training accuracy → 99%
Test accuracy     → 72%
```

This is **overfitting**.

The model has learned the training data too specifically instead of learning patterns that generalize.

### Goal of regularization

> **Encourage better generalization and reduce overfitting.**

---

# 2. Regularization Objective

A general regularized objective can be written as:

$$
J(\theta)=L(\theta)+\lambda R(\theta)
$$

where:

* $L(\theta)$ → training loss
* $R(\theta)$ → regularization term
* $\lambda$ → regularization strength

Common techniques:

```text
Regularization
├── Weight decay
├── Dropout
├── Early stopping
├── Data augmentation
└── L1/L2 penalties
```

---

# 3. Base MLP

Example baseline:

```python
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)
```

Architecture:

```text
10 → 64 → 32 → 16 → 2
```

We can now introduce regularization techniques.

---

# 4. Weight Decay 🔥🔥🔥

Weight decay encourages model parameters to have smaller magnitudes.

With AdamW, conceptually:

$$
\theta_t
=
(1-\eta\lambda)\theta_{t-1}
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
$$

The important practical idea:

> **Weight decay directly shrinks parameters during optimization.**

### PyTorch

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

Without weight decay:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

With weight decay:

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

---

# 5. Weight Decay Intuition

Suppose the model learns:

```text
w₁ = 8.7
w₂ = -6.4
w₃ = 0.03
```

Weight decay encourages parameters toward smaller magnitudes.

Conceptually:

```text
Large weights
     ↓
Shrink
     ↓
Simpler parameterization
     ↓
Potentially better generalization
```

It does **not** mean every weight becomes zero.

---

# 6. Dropout 🔥🔥🔥

Dropout randomly removes some activations during training.

Before:

```text
● ● ● ● ● ● ● ●
```

After dropout:

```text
● ✕ ● ● ✕ ● ✕ ●
```

The dropped activations are temporarily inactive for that training step.

### PyTorch

```python
nn.Dropout(0.3)
```

means:

> Drop probability = 30%.

Example:

```python
class RegularizedMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)
```

---

# 7. Dropout During Training vs Inference 🔥🔥🔥

### Training

```python
model.train()
```

```text
Dropout → ON
```

### Evaluation

```python
model.eval()
```

```text
Dropout → OFF
```

PyTorch handles this automatically.

### Important distinction

```python
model.eval()
```

changes the behavior of layers such as Dropout and BatchNorm.

```python
with torch.no_grad():
```

disables gradient tracking.

They are **not the same thing**.

---

# 8. Dropout Intuition

Without dropout:

```text
Neuron A
Neuron B
Neuron C
Neuron D
     ↓
Network may rely heavily on specific paths
```

With dropout:

```text
Some neurons randomly disappear
          ↓
Network cannot always rely on
the same neurons
```

This encourages more distributed representations.

---

# 9. Batch Normalization

BatchNorm normalizes activations using mini-batch statistics.

$$
\mu_B=\frac{1}{m}\sum_i x_i
$$

$$
\sigma_B^2=
\frac{1}{m}
\sum_i(x_i-\mu_B)^2
$$

Normalize:

$$
\hat{x}_i=
\frac{x_i}
{\sqrt{\sigma_B^2+\epsilon}}
$$

More precisely:

$$
\hat{x}_i=
\frac{x_i-\mu_B}
{\sqrt{\sigma_B^2+\epsilon}}
$$

Then learn scale and shift:

$$
y_i=\gamma\hat{x}_i+\beta
$$

---

# 10. BatchNorm in an MLP

```python
class RegularizedMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(10, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.network(x)
```

Common pattern:

```text
Linear
 ↓
BatchNorm
 ↓
Activation
```

BatchNorm can improve optimization and may also have regularizing effects.

> **Don't think of BatchNorm as simply another form of Dropout.**

---

# 11. Dropout + BatchNorm

They can be used together.

For example:

```text
Linear
 ↓
BatchNorm
 ↓
ReLU
 ↓
Dropout
```

But:

> Don't automatically add every regularization technique to every model.

The appropriate architecture depends on the problem and experiments.

---

# 12. Early Stopping 🔥🔥🔥

Suppose:

| Epoch | Training Loss | Validation Loss |
| ----: | ------------: | --------------: |
|     1 |          0.60 |            0.62 |
|     2 |          0.45 |            0.48 |
|     3 |          0.31 |            0.35 |
|     4 |          0.22 |            0.29 |
|     5 |          0.15 |            0.31 |
|     6 |          0.10 |            0.36 |
|     7 |          0.07 |            0.41 |

Training loss keeps decreasing:

```text
Train Loss ↓↓↓↓↓
```

while validation loss starts increasing:

```text
Validation Loss ↓
                 ↑
                 ↑
```

This can indicate overfitting.

### Early stopping

> Stop training when validation performance stops improving.

---

# 13. Patience

Suppose:

```text
patience = 3
```

Best result occurs at epoch 10.

Then:

```text
Epoch 11 → no improvement
Epoch 12 → no improvement
Epoch 13 → no improvement
             ↓
           Stop
```

The exact implementation can vary, but the idea is:

> Give the model several chances to improve before stopping.

---

# 14. Regularized MLP

A possible regularized architecture:

```text
Input
  ↓
Linear
  ↓
BatchNorm
  ↓
ReLU
  ↓
Dropout
  ↓
Linear
  ↓
BatchNorm
  ↓
ReLU
  ↓
Dropout
  ↓
Output
```

Optimizer:

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

Loss:

```python
loss_fn = nn.CrossEntropyLoss()
```

---

# 15. Training Loop 🔥🔥🔥

```python
for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        logits = model(X_batch)

        loss = loss_fn(
            logits,
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
```

The important point:

> Regularization changes the model/optimizer configuration, but the fundamental training loop remains the same.

```text
Forward
   ↓
Loss
   ↓
Backward
   ↓
Gradients
   ↓
Optimizer
   ↓
Parameter Update
```

---

# 16. Validation

After each epoch:

```python
model.eval()

with torch.no_grad():

    val_logits = model(X_val)

    val_loss = loss_fn(
        val_logits,
        y_val
    )
```

Monitor:

```text
Training loss
Validation loss
Training accuracy
Validation accuracy
```

---

# 17. Recognizing Overfitting 🔥🔥🔥

### Healthy training

```text
Training loss    ↓
Validation loss  ↓
```

Both improve.

### Possible overfitting

```text
Training loss    ↓↓↓
Validation loss      ↑
```

Training continues improving while validation performance deteriorates.

### Possible underfitting

```text
Training loss   → high
Validation loss → high
```

The model may be too simple, insufficiently trained, or otherwise unable to adequately fit the task.

---

# 18. Different Regularization Techniques

| Technique             | Main mechanism                                           |
| --------------------- | -------------------------------------------------------- |
| **Weight decay**      | Shrinks parameters                                       |
| **Dropout**           | Randomly removes activations during training             |
| **BatchNorm**         | Normalizes activations and changes optimization behavior |
| **Early stopping**    | Limits training based on validation performance          |
| **Data augmentation** | Creates useful training variation                        |

These techniques are **not interchangeable**.

---

# 19. Very Important Distinction 🔥🔥🔥

### Weight Decay

Acts primarily on:

> **Parameters / weights**

```text
Weights
  ↓
Shrink
```

### Dropout

Acts on:

> **Activations**

```text
Activations
  ↓
Randomly zero some
```

### BatchNorm

Acts on:

> **Activation statistics**

```text
Activations
  ↓
Normalize
  ↓
Scale + Shift
```

### Early Stopping

Acts on:

> **Training duration**

```text
Training
   ↓
Validation stops improving
   ↓
Stop
```

---

# 20. Experiments 🧪

This module is also about **experimenting**, not simply writing one model.

You can compare:

### Experiment 1 — Baseline

```text
Plain MLP
```

### Experiment 2

```text
MLP + Dropout
```

### Experiment 3

```text
MLP + Weight Decay
```

### Experiment 4

```text
MLP + BatchNorm
```

### Experiment 5

```text
MLP + Dropout + Weight Decay
```

Then record:

| Model        | Train Acc. | Val Acc. | Val Loss |
| ------------ | ---------: | -------: | -------: |
| Plain MLP    |          — |        — |        — |
| Dropout      |          — |        — |        — |
| Weight Decay |          — |        — |        — |
| BatchNorm    |          — |        — |        — |

The goal is to **observe the effect experimentally**, rather than assuming one technique always performs best.

---

# 21. Hyperparameters to Experiment With 🔥

### Dropout

```python
nn.Dropout(0.1)
nn.Dropout(0.3)
nn.Dropout(0.5)
```

### Learning rate

```python
lr=0.0001
lr=0.001
lr=0.01
```

### Weight decay

```python
weight_decay=0
weight_decay=0.001
weight_decay=0.01
```

### Batch size

```text
16
32
64
128
```

### Hidden-layer width

```text
32
64
128
256
```

When learning the effect of a variable:

> **Change one important variable at a time.**

This makes the experiment easier to interpret.

---

# 22. Too Much Regularization 🚨

Regularization is not automatically beneficial at higher strength.

Too much can produce **underfitting**.

For example:

```text
Very high dropout
+
Strong weight decay
+
Small model
```

could make the model unable to learn the training patterns sufficiently.

Therefore:

> **The goal is appropriate regularization, not maximum regularization.**

---

# 23. Common Mistakes 🚨

### ❌ Forgetting `model.eval()`

For a model containing Dropout or BatchNorm:

```python
model.eval()
```

should be used during validation/inference.

---

### ❌ Thinking `eval()` disables gradients

It doesn't.

```python
model.eval()
```

changes layer behavior.

For disabling gradient tracking:

```python
with torch.no_grad():
```

---

### ❌ Thinking BatchNorm is just regularization

BatchNorm primarily normalizes activations and affects optimization. It can have regularizing effects, but its role is broader than simply preventing overfitting.

---

### ❌ Excessive dropout

For example:

```python
nn.Dropout(0.9)
```

can remove too many activations and make learning difficult.

---

### ❌ Assuming every regularization method must be used

You don't need:

```text
Dropout
+
BatchNorm
+
Weight decay
+
Early stopping
```

in every model.

Choose and evaluate techniques according to the problem.

---

# 24. Complete Regularized MLP Mental Model 🧠

```text
                    MLP
                     ↓
              Regularization
                     │
       ┌─────────────┼─────────────┐
       ↓             ↓             ↓
 Weight Decay     Dropout       BatchNorm
       ↓             ↓             ↓
 Smaller         Randomly      Normalize
 Parameters       remove        activations
                  activations
       └─────────────┼─────────────┘
                     ↓
              Validation Monitor
                     ↓
               Early Stopping
                     ↓
              Better Generalization
```

---

# 🎯 Interview / GATE Points

### What is regularization?

A set of techniques used to reduce overfitting and improve generalization.

### What does weight decay do?

Encourages smaller parameter magnitudes.

### What does Dropout do?

Randomly sets a fraction of activations to zero during training.

### Is Dropout active during inference?

**No.**

`model.eval()` disables its training behavior.

### What does BatchNorm do?

Normalizes activations using batch statistics during training and uses running statistics during inference.

### What does early stopping monitor?

Usually validation performance/loss.

### Can excessive regularization hurt?

**Yes.** It can cause underfitting.

---

# 🧠 Final Cheat Sheet

```text
REGULARIZED MLP
──────────────────────────────────

Goal:
Reduce overfitting
Improve generalization

Weight Decay:
→ Shrinks parameters

Dropout:
→ Randomly drops activations during training

BatchNorm:
→ Normalizes activations
→ Learnable γ and β

Early Stopping:
→ Monitor validation performance
→ Stop when improvement stops

Training:
model.train()

Validation:
model.eval()
torch.no_grad()

Core loop:
Forward
   ↓
Loss
   ↓
Backward
   ↓
Gradients
   ↓
Optimizer
   ↓
Update

Main warning:
Too much regularization → underfitting
```

### 🔥 One-line memory trick

> **Weight decay shrinks weights → Dropout removes activations → BatchNorm normalizes activations → Early stopping limits training.**

---
